# Imports

In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from rdkit import Chem

PROJECT_ROOT = Path.cwd().parent  # si el notebook esta en /notebooks
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths_and_constants import Paths
_P = Paths()

# Functions

In [2]:
def is_ion_smiles(smiles: str) -> bool:
    """
    Returns true if SMILES corresponds to a monoatomic ion.
    """
    if smiles is None or (isinstance(smiles, float) and pd.isna(smiles)):
        return False

    m = Chem.MolFromSmiles(str(smiles))
    if m is None:
        return False

    # Ion = un solo átomo, sin enlaces
    if m.GetNumAtoms() != 1:
        return False

    atom = m.GetAtomWithIdx(0)
    return atom.GetSymbol().upper() in ion_symbols

def add_negatives(
    df,
    seq_col="protein_sequence",
    lig_col="smiles",
    neg_per_pos=1,
    seed=42,
    max_tries=1000,
):
    rng = np.random.default_rng(seed)

    df = df.copy()
    if "label" not in df.columns:
        df["label"] = 1

    positives = set(zip(df[seq_col], df[lig_col]))
    ligands_filtered = set(df[lig_col].dropna().unique())

    negatives = set()

    for p, grp in df.groupby(seq_col):
        pos_ligands_p = set(grp[lig_col].dropna().unique())
        if not pos_ligands_p:
            continue

        candidate_ligands = list(ligands_filtered - pos_ligands_p)
        n_neg = neg_per_pos * len(pos_ligands_p)
        if not candidate_ligands:
            continue

        if len(candidate_ligands) >= n_neg:
            chosen = rng.choice(candidate_ligands, size=n_neg, replace=False)
        else:
            chosen = rng.choice(candidate_ligands, size=n_neg, replace=True)

        for l_neg in chosen:
            if (p, l_neg) not in positives:
                negatives.add((p, l_neg))

        if len(candidate_ligands) < n_neg and max_tries > 0:
            tries = 0
            while len(negatives) < n_neg and tries < max_tries:
                l_neg = rng.choice(candidate_ligands)
                if (p, l_neg) not in positives:
                    negatives.add((p, l_neg))
                tries += 1

    neg_df = pd.DataFrame(list(negatives), columns=[seq_col, lig_col])
    neg_df["label"] = 0

    # columns alignment
    for col in df.columns:
        if col not in neg_df.columns:
            neg_df[col] = np.nan
    neg_df = neg_df[df.columns]

    out_df = pd.concat([df, neg_df], ignore_index=True)
    return out_df

# Dataprep for Biolip

In [3]:
biol = pd.read_csv(_P.data_interim_dir / "biolip" / "BioLiP_nr.csv")
print(biol.shape)
print(biol.columns)
biol = biol[(biol['ec_number'].notna())]
print(biol.shape)
biol = biol[(biol['uniprot_id'].notna())]
print(biol.shape)
biol = biol[(biol['ligand_id'].notna())]
print(biol.shape)

(86458, 21)
Index(['pdb_id', 'chain_id', 'resolution', 'binding_site_id', 'ligand_id',
       'ligand_chain_id', 'ligand_serial_number', 'binding_site_residues',
       'nearby_residues', 'ligand_contact_residues', 'catalytic_site_residues',
       'ec_number', 'go_terms', 'binding_affinity_1', 'binding_affinity_2',
       'binding_affinity_3', 'binding_affinity_4', 'uniprot_id', 'pubmed_id',
       'binding_site_range', 'protein_sequence'],
      dtype='object')
(68497, 21)
(68291, 21)
(68206, 21)


/tmp/ipykernel_9141/1044276033.py:1: DtypeWarning: Columns (13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  biol = pd.read_csv(_P.data_interim_dir / "biolip" / "BioLiP_nr.csv")


In [4]:
ccd = pd.read_csv(
    _P.data_interim_dir / 'biolip' / "Components-smiles-cactvs.smi",
    sep="\t",
    header=None,
    names=["smiles", "ccd_id", "ligand_name"],
    dtype=str
)
ccd["ccd_id"] = ccd["ccd_id"].str.strip()
ccd["smiles"] = ccd["smiles"].str.strip()
ccd["ligand_name"] = ccd["ligand_name"].str.strip()

# Lookups (diccs) to map ccd_id -> (smiles, name)
ccd_exact = ccd.drop_duplicates("ccd_id").set_index("ccd_id")[["smiles", "ligand_name"]]
ccd_upper = (
    ccd.assign(ccd_id_upper=ccd["ccd_id"].str.upper())
       .drop_duplicates("ccd_id_upper")
       .set_index("ccd_id_upper")[["smiles", "ligand_name"]]
)

# Prepare BioLiP IDs
biol2 = biol.copy()
biol2["ligand_id_raw"] = biol2["ligand_id"].astype(str).str.strip()

# Map (case-sensitive)
biol2["smiles"] = biol2["ligand_id_raw"].map(ccd_exact["smiles"])
biol2["ligand_name"] = biol2["ligand_id_raw"].map(ccd_exact["ligand_name"])
biol2["map_status"] = "exact"

# Fallback to UPPER (only if not mapped yet and not polymer token)
polymer_tokens = {"dna", "rna", "pep", "peptide", "protein", "polypeptide"}

need = biol2["smiles"].isna()
ok = ~biol2["ligand_id_raw"].str.lower().isin(polymer_tokens)

idx = biol2.index[need & ok]
biol2.loc[idx, "smiles"] = biol2.loc[idx, "ligand_id_raw"].str.upper().map(ccd_upper["smiles"])
biol2.loc[idx, "ligand_name"] = biol2.loc[idx, "ligand_id_raw"].str.upper().map(ccd_upper["ligand_name"])
biol2.loc[idx, "map_status"] = "rescued_upper"

# Track whatever was blocked and not mapped
blocked = need & ~ok
biol2.loc[blocked, "map_status"] = "polymer_token_blocked"

still_unmapped = biol2["smiles"].isna() & ok
biol2.loc[still_unmapped, "map_status"] = "unmapped"
biol_smiles = biol2
print(biol_smiles.shape)
biol_smiles.dropna(subset=["smiles"], inplace=True)
print(biol_smiles.shape)
biol_smiles

(68206, 25)
(55291, 25)


,pdb_id,chain_id,resolution,binding_site_id,ligand_id,ligand_chain_id,ligand_serial_number,binding_site_residues,nearby_residues,ligand_contact_residues,...,binding_affinity_3,binding_affinity_4,uniprot_id,pubmed_id,binding_site_range,protein_sequence,ligand_id_raw,smiles,ligand_name,map_status
2,10mh,A,2.550,BS03,SAH,A,1,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,C81 E119 R163 R165,...,NaN,NaN,P05102,9925782.0,328,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...,SAH,N[CH](CCSC[CH]1O[CH]([CH](O)[CH]1O)n2cnc3c(N)n...,s-adenosyl-l-homocysteine,exact
3,11as,A,2.500,BS01,ASN,A,1,D46 A74 Q116 Y218 S251 R255,D43 A71 Q113 Y215 S248 R252,D46 R100 Q116 D235 E248 S251,...,NaN,NaN,P00963,9437423.0,331,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,ASN,N[CH](CC(N)=O)C(O)=O,asparagine,exact
4,11ba,A,2.060,BS01,UPA,A,1,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,H12 K41 H119 F120 D121,...,NaN,NaN,P00669,10082366.0,125,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...,UPA,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[CH]4[...,uridylyl-2'-5'-phospho-adenosine,exact
5,13pk,A,2.500,BS01,ADP,A,1,G217 A218 K223 L315 P340 G342 V343 E345 G375 D...,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,R39 K219 G376 G399,...,NaN,NaN,P07378,9000079.0,421,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,ADP,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[P](O)...,adenosine-5'-diphosphate,exact
6,155c,A,2.500,BS01,HEM,A,1,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,NaN,...,NaN,NaN,P00096,180013.0,135,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...,HEM,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=C...,protoporphyrin ix containing fe,exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82359,9ezj,C,2.612,BS01,ZIQ,C,1,R103 E105 W208 R211 T212 P213 F304,R64 E66 W148 R151 T152 P153 F244,NaN,...,NaN,NaN,P48775,39537789.0,501,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,ZIQ,C[C](N)(Cc1c[nH]c2ccccc12)C(O)=O,alpha-methyl-l-tryptophan,exact
82360,9ezj,C,2.612,BS02,A1H8G,C,1,F72 H76 F129 F140 F153 Q154 S155 H328,F33 H37 F90 F101 F111 Q112 S113 H268,NaN,...,NaN,NaN,P48775,39537789.0,502,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,A1H8G,CCOC(=O)C1=C(OC)N=C2[CH](CCCN2C1=O)N([CH](C)c3...,ethyl (9~{r})-2-methoxy-4-oxidanylidene-9-[[(1...,exact
82553,9fgp,A,1.490,BS01,ZN,A,1,E125 D164,E36 D75,NaN,...,NaN,NaN,Q6V702,NaN,301,SMDNFLTALAMREEDNRSGKLSSVIFIRDRNSHGQEISGYIDYAHR...,ZN,[Zn++],zinc ion,exact
86244,9pcy,A,-1.000,BS01,CU,A,1,H37 C84 H87,H37 C84 H87,H37 C84 H87 M92,...,NaN,NaN,P00287,1920431.0,100,LEVLLGSGDGSLVFVPSEFSVPSGEKIVFKNNAGFPHNVVFDEDEI...,CU,[Cu++],copper (ii) ion,exact


In [5]:
ion_symbols = {
    "LI","NA","K","RB","CS",
    "MG","CA","SR","BA",
    "ZN","FE","CU","MN","CO","NI",
    "CD","HG","AL","CR","V","MO","W",
    "CL","BR","I","F",
    "AG","AU","PT","PB","SN"
}

# Ion mask
is_ion = biol_smiles["smiles"].apply(is_ion_smiles)

# final filter
biol_filtered = biol_smiles[~is_ion].copy()

print("original:", biol_smiles.shape)
print("removed ions:", int(is_ion.sum()))
print("kept:", biol_filtered.shape)

# sanity check
print("\nExamples removed as ions:")
print(
    biol_smiles.loc[is_ion, ["ligand_id_raw", "ligand_name", "smiles"]]
    .head(10)
)

[00:51:35] SMILES Parse Error: syntax error while parsing: CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
[00:51:35] SMILES Parse Error: check for mistakes around position 29:
[00:51:35] C(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=
[00:51:35] ~~~~~~~~~~~~~~~~~~~~^
[00:51:35] SMILES Parse Error: Failed parsing SMILES 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C' for input: 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C'
[00:51:35] SMILES Parse Error: syntax error while parsing: CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
[00:51:35] SMILES Parse Error: check for mistakes around position 29:
[00:51:35] C(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=
[00:51:35] ~~~~~~~~~~~~~~~~~~~~^
[00:51:35] SMILES Parse Error: Failed parsing SMILES 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O

original: (55291, 25)
removed ions: 18951
kept: (36340, 25)

Examples removed as ions:
   ligand_id_raw         ligand_name  smiles
21            CO     cobalt (ii) ion  [Co++]
22            CO     cobalt (ii) ion  [Co++]
23            MN  manganese (ii) ion  [Mn++]
24            MN  manganese (ii) ion  [Mn++]
25            CO     cobalt (ii) ion  [Co++]
26            CO     cobalt (ii) ion  [Co++]
30            CA         calcium ion  [Ca++]
33            CA         calcium ion  [Ca++]
50            MN  manganese (ii) ion  [Mn++]
57            CA         calcium ion  [Ca++]


[00:51:38] SMILES Parse Error: syntax error while parsing: CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
[00:51:38] SMILES Parse Error: check for mistakes around position 29:
[00:51:38] C(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=
[00:51:38] ~~~~~~~~~~~~~~~~~~~~^
[00:51:38] SMILES Parse Error: Failed parsing SMILES 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C' for input: 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C'
[00:51:38] SMILES Parse Error: syntax error while parsing: CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O)C(=C8C=C)C)c(C)c7C=C
[00:51:38] SMILES Parse Error: check for mistakes around position 29:
[00:51:38] C(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=
[00:51:38] ~~~~~~~~~~~~~~~~~~~~^
[00:51:38] SMILES Parse Error: Failed parsing SMILES 'CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=CC8=N|6C(=Cc4c(C)c3CCC(O)=O

In [6]:
biol_filtered

,pdb_id,chain_id,resolution,binding_site_id,ligand_id,ligand_chain_id,ligand_serial_number,binding_site_residues,nearby_residues,ligand_contact_residues,...,binding_affinity_3,binding_affinity_4,uniprot_id,pubmed_id,binding_site_range,protein_sequence,ligand_id_raw,smiles,ligand_name,map_status
2,10mh,A,2.550,BS03,SAH,A,1,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,F18 A19 G20 L21 G23 E40 W41 D60 N304 S305,C81 E119 R163 R165,...,NaN,NaN,P05102,9925782.0,328,MIEIKDKQLTGLRFIDLFAGLGGFRLALESCGAECVYSNEWDKYAQ...,SAH,N[CH](CCSC[CH]1O[CH]([CH](O)[CH]1O)n2cnc3c(N)n...,s-adenosyl-l-homocysteine,exact
3,11as,A,2.500,BS01,ASN,A,1,D46 A74 Q116 Y218 S251 R255,D43 A71 Q113 Y215 S248 R252,D46 R100 Q116 D235 E248 S251,...,NaN,NaN,P00963,9437423.0,331,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,ASN,N[CH](CC(N)=O)C(O)=O,asparagine,exact
4,11ba,A,2.060,BS01,UPA,A,1,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,K41 V43 N44 T45 N67 Q69 N71 A109 H119 F120,H12 K41 H119 F120 D121,...,NaN,NaN,P00669,10082366.0,125,KESAAAKFERQHMDSGNSPSSSSNYCNLMMCCRKMTQGKCKPVNTF...,UPA,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[CH]4[...,uridylyl-2'-5'-phospho-adenosine,exact
5,13pk,A,2.500,BS01,ADP,A,1,G217 A218 K223 L315 P340 G342 V343 E345 G375 D...,G213 A214 K219 L311 P336 G338 V339 E341 G371 D...,R39 K219 G376 G399,...,NaN,NaN,P07378,9000079.0,421,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,ADP,Nc1ncnc2n(cnc12)[CH]3O[CH](CO[P](O)(=O)O[P](O)...,adenosine-5'-diphosphate,exact
6,155c,A,2.500,BS01,HEM,A,1,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,NaN,...,NaN,NaN,P00096,180013.0,135,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...,HEM,CC1=C(CCC(O)=O)C2=Cc3n4[Fe]5|6|N2=C1C=c7n5c(=C...,protoporphyrin ix containing fe,exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82210,9exw,A,2.430,BS01,A1H7X,A,1,V230 Y233 W236 F266 G268 A270 E272 Q321,V14 Y17 W20 F50 G52 A54 E56 Q105,NaN,...,NaN,NaN,O96028,38748070.0,403,LLKYNVGDLVWSKVSGYPWWPCMVSADPLLHSYTKLKGQKKSARQY...,A1H7X,Cn1cnc(c2cc(CNc3cccnc3)ccc2C)c1c4ccc5NC(=O)COc5c4,7-[3-methyl-5-[2-methyl-5-[(pyridin-3-ylamino)...,exact
82211,9exy,B,1.699,BS01,A1H7Z,B,1,V230 Y233 W236 F266 F267 G268 A270 E272 Q321,V17 Y20 W23 F53 F54 G55 A57 E59 Q91,NaN,...,NaN,NaN,O96028,38748070.0,401,KDHLLKYNVGDLVWSKVSGYPWWPCMVSADPLLHSYTKLKGQAASA...,A1H7Z,Cc1onc(c2cc(CN3CCCCC3)ccc2C)c1c4ccc5NC(=O)COc5c4,7-[5-methyl-3-[2-methyl-5-(piperidin-1-ylmethy...,exact
82359,9ezj,C,2.612,BS01,ZIQ,C,1,R103 E105 W208 R211 T212 P213 F304,R64 E66 W148 R151 T152 P153 F244,NaN,...,NaN,NaN,P48775,39537789.0,501,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,ZIQ,C[C](N)(Cc1c[nH]c2ccccc12)C(O)=O,alpha-methyl-l-tryptophan,exact
82360,9ezj,C,2.612,BS02,A1H8G,C,1,F72 H76 F129 F140 F153 Q154 S155 H328,F33 H37 F90 F101 F111 Q112 S113 H268,NaN,...,NaN,NaN,P48775,39537789.0,502,LIYGNYLHLEKVLNAQELQSETKGNKIHDEHLFIITHQAYELWFKQ...,A1H8G,CCOC(=O)C1=C(OC)N=C2[CH](CCCN2C1=O)N([CH](C)c3...,ethyl (9~{r})-2-methoxy-4-oxidanylidene-9-[[(1...,exact


In [7]:
neg_df = add_negatives(
    biol_filtered,
    seq_col="protein_sequence",
    lig_col="smiles",
    neg_per_pos=1,
    seed=42,
    max_tries=1000,
)

KeyboardInterrupt: 

In [ ]:
neg_df.to_csv(
    _P.data_processed_biolip_dir / "biolip_cleaned_with_negatives.csv",
    index=False
)

# Dataprep for DDG and ThermoStab

In [3]:
fireprot = pd.read_csv(_P.data_raw_fireprotdb_dir / "fireprotdb_ec.tsv", sep="\t")

# Coerce to numeric to drop non-numeric placeholders if any
fireprot["DDG"] = pd.to_numeric(fireprot["DDG"], errors="coerce")
fireprot["DTM"] = pd.to_numeric(fireprot["DTM"], errors="coerce")

ddg_df = fireprot[fireprot["DDG"].notna()].copy()
print(ddg_df.shape)
dtm_df = fireprot[fireprot["DTM"].notna()].copy()
print(dtm_df.shape)

(3992, 53)
(3514, 53)


/tmp/ipykernel_33333/2487166406.py:1: DtypeWarning: Columns (12,15,22,24,27,29,30,32,33,34,35,41,42,43,44,45,46,47,52) have mixed types. Specify dtype option on import or set low_memory=False.
  fireprot = pd.read_csv(_P.data_raw_fireprotdb_dir / "fireprotdb_ec.tsv", sep="\t")


In [4]:
ddg_df.to_csv(_P.data_processed_properties / "ddg_mut" / "fireprotdb_ddg.csv", index=False)
dtm_df.to_csv(_P.data_processed_properties / "dtm" / "fireprotdb_dtm.csv", index=False)

# Dataprep for phOpt (BRENDA)

In [59]:
import json

with open("../data/raw/brenda/brenda_2025_1.json", "r") as f:
    data = json.load(f)

def search_for_uniprot(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if "uni" in k.lower() or "access" in k.lower():
                print("FOUND KEY:", k)
            search_for_uniprot(v)
    elif isinstance(obj, list):
        for item in obj:
            search_for_uniprot(item)

search_for_uniprot(data)

def find_first_accession(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_path = f"{path} -> {k}"
            if k == "accessions":
                print("FOUND at:", new_path)
                print("VALUE:", v)
                return True
            if find_first_accession(v, new_path):
                return True
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            if find_first_accession(item, f"{path}[{i}]"):
                return True
    return False

find_first_accession(data)

FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY: accessions
FOUND KEY:

True

In [ ]:
brenda_data = data["data"]

ec_with_ph = next(
    ec for ec, ec_data in brenda_data.items()
    if isinstance(ec_data, dict) and "ph_optimum" in ec_data
)

ec_data = brenda_data[ec_with_ph]

print("ec:", ec_with_ph)
print("keys:", ec_data.keys())
print("ph_optimum[0]:", ec_data["ph_optimum"][0])
protein_id = ec_data["ph_optimum"][0]["proteins"][0]

print(ec_data["protein"][protein_id])
list(ec_data["protein"].values())[0]
list(data.keys())[:5]
brenda_data = data["data"]
ec_example = next(iter(brenda_data.values()))
print(ec_example.keys())
ec_with_ph = next(
    ec for ec, ec_data in brenda_data.items()
    if isinstance(ec_data, dict) and "ph_optimum" in ec_data
)
ec_data = brenda_data[ec_with_ph]
ec_data.keys()
ec = ec_with_ph  # o cualquier EC que estés viendo
ec_data = brenda_data[ec]

for pid, pdata in ec_data["protein"].items():
    print(pid, pdata.keys())
    break

In [60]:
results = []

for ec, ec_data in brenda_data.items():
    if not isinstance(ec_data, dict) or "ph_optimum" not in ec_data:
        continue

    proteins = ec_data.get("protein", {})

    for entry in ec_data["ph_optimum"]:
        ph = entry.get("value")
        protein_ids = entry.get("proteins", [])

        for pid in protein_ids:
            p = proteins.get(str(pid), {})  # pid puede venir como '10'
            org = p.get("organism")

            accs = p.get("accessions", [])
            uniprot = accs[0] if accs else None  # si hay varios, tomas el primero

            results.append({
                "ec": ec,
                "protein_id": str(pid),
                "organism": org,
                "phopt": ph,
                "uniprot_from_brenda": uniprot,
            })

df_raw = pd.DataFrame(results)
total = len(df_raw)

numeric = pd.to_numeric(df_raw["phopt"], errors="coerce").notna().sum()

print("Total entries:", total)
print("Numeric entries:", numeric)
print("Non-numeric entries:", total - numeric)
print("Fraction non-numeric:", (total - numeric) / total)
non_numeric = df_raw[pd.to_numeric(df_raw["phopt"], errors="coerce").isna()]["phopt"].unique()
print("Non-numeric values:", non_numeric[:20])

Total entries: 53256
Numeric entries: 44175
Non-numeric entries: 9081
Fraction non-numeric: 0.17051599819738622
Non-numeric values: ['7.7-8.6' '6.7-7.3' '6-7' '8-9' '5.5-6' '6.5-7' '8.6-8.8' '8-8.5'
 '8.5-9.1' '6.9-7.5' '9-10' '8-8.4' '5.6-6.5' '8.5-8.8' '10-10.5' '7-7.5'
 '6.7-7' '10-11' '5.7-6.2' '6.5-7.5']


In [61]:
import re

def parse_ph(x):
    if pd.isna(x):
        return None
    
    x = str(x)
    
    # case range tipe a-b
    if "-" in x:
        parts = x.split("-")
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except:
            return None
    
    # case simple
    try:
        return float(x)
    except:
        return None


df_raw["phopt"] = df_raw["phopt"].apply(parse_ph)
print(df_raw.shape)

(53256, 5)


In [66]:
df_raw["phopt"] = df_raw["phopt"].apply(parse_ph)
df_raw = df_raw.dropna(subset=["phopt"])

df_one = (
    df_raw.groupby(["ec", "protein_id", "organism", "uniprot_from_brenda"], as_index=False)
      .agg(
          phopt_median=("phopt", "median"),
          n_measurements=("phopt", "count"),
          phopt_std=("phopt", "std")
      )
)
df_one.to_csv(_P.data_interim_dir / 'phopt' / 'brenda_phopt.csv', index=False)